# Q3 · Nowcasting the next CPI print

In [1]:
%load_ext autoreload
%autoreload 2
import logging, sys, warnings
import numpy as np, pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=RuntimeWarning)
logging.basicConfig(level=logging.WARNING, force=True)
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.precision", 3, "display.width", 160)

import nowcast as nc
import truflation_components as tc
sys.path.insert(0, "../Q1_Q2")
import pipeline as pl
import charts as ch                      # same dark theme / colours as the Q2 dashboard

inp = nc.load_inputs()                   # stored Q1 pipeline output + CPI history + Truflation components
res = pl.load_cached()
feeds = pd.read_parquet("data/truflation_feeds.parquet")
mirrors = pd.read_csv("data/bls_mirrors.csv")
asof_c, comp_w = tc.build_components(asof=True, release=inp.release)
vint_c, _ = tc.build_components(asof=False)
print(f"Truflation through {inp.truf_pub.index.max():%d %b %Y} | last CPI print {inp.cpi.last_valid_index():%b %Y} "
      f"| next release {inp.release[pd.Timestamp('2026-10-01')]:%d %b %Y}")

Truflation through 21 Oct 2026 | last CPI print Sep 2026 | next release 10 Nov 2026


In [2]:
# ---- backtests (walk-forward, both origin conventions) and the live nowcast --------------------------------
ft19 = nc.feature_table(19, inp); bt19 = nc.walk_forward(ft19, inp)
ftev = nc.feature_table("eve", inp); btev = nc.walk_forward(ftev, inp)
live = nc.live_nowcast(inp, ft19, horizon=19)
CENTRAL = "Interpolated Oct-25 (geometric mean of Sep & Nov)"
BEST = "Truflation components"
err = (bt19[BEST] - bt19["y"]).dropna()
q_lo, q_hi = err.loc["2021-12":].quantile([0.1, 0.9])            # empirical 80% band, live period
c = live[CENTRAL]; row = c["row"]; p = c["pred"][BEST]
sc_full = nc.scorecard(bt19, nc.PERIODS["Full OOS (2014–)"])
sc_live = nc.scorecard(bt19, nc.PERIODS["Truflation live (Dec-2021–)"])

## Bottom line

In [3]:
contrib = c["contrib"][BEST].rename({"Intercept": "Model intercept", "BE": "Base effect", "dFood": "Food",
                                     "dEnergy": "Energy (fuel + utilities)", "dHousing": "Housing", "dRest": "Everything else"})
scen = pd.Series({k: v["pred"][BEST] for k, v in live.items()})
top = contrib.drop("Model intercept").sort_values(key=abs, ascending=False)
m = res.monthly
r_lvl, r_chg = sc_full.loc[BEST, "RMSE (pp)"], sc_full.loc["Naive (last print)", "RMSE (pp)"]
display(Markdown(f'''
**Oct-26 CPI (out {inp.release[pd.Timestamp("2026-10-01")]:%d %b}): {p:.2f}% YoY**, up from {row["yL"]:.2f}% · 80% range {p + q_lo:.2f}–{p + q_hi:.2f}%
- **Driver:** energy {contrib["Energy (fuel + utilities)"]:+.2f}pp (Truflation energy +{asof_c["grp_Energy"].iloc[-1]:.0f}% YoY), base effect {contrib["Base effect"]:+.2f}pp; food and housing subdued.
- **Track record:** {r_lvl:.2f}pp error vs {r_chg:.2f}pp for "no change"; direction right {sc_full.loc[BEST, "Direction hit"]:.0%} of the time.
- **Use for direction, not level:** the gap to CPI was {m["spread"].dropna().iloc[-1]:+.2f}pp in September.
- **Caveat:** ~{mirrors["weight"].sum():.0f}% of Truflation is copied BLS data, so it partly freezes when BLS stops publishing.
'''))


**Oct-26 CPI (out 10 Nov): 3.99% YoY**, up from 3.74% · 80% range 3.72–4.33%
- **Driver:** energy +0.14pp (Truflation energy +14% YoY), base effect +0.07pp; food and housing subdued.
- **Track record:** 0.18pp error vs 0.39pp for "no change"; direction right 90% of the time.
- **Use for direction, not level:** the gap to CPI was -1.35pp in September.
- **Caveat:** ~30% of Truflation is copied BLS data, so it partly freezes when BLS stops publishing.


## What Truflation is made of
~30% of the basket is copied BLS data; only ~6% updates daily.

In [4]:
fw = feeds.assign(kind=np.where(feeds["bls_mirror"], "BLS CPI mirror (monthly)", feeds["freq"].str.capitalize() + " alternative data"))
summary = fw.groupby("kind").agg(feeds=("weight", "size"), basket_weight=("weight", "sum")).sort_values("basket_weight", ascending=False)
summary["basket_weight"] = summary["basket_weight"].round(1)
display(summary)
display(mirrors.rename(columns={"bls_match": "matches BLS series", "corr": "MoM corr", "weight": "basket weight %"})
        .set_index("leaf").head(12))

,feeds,basket_weight
kind,,
Monthly alternative data,78,52.4
BLS CPI mirror (monthly),22,30.4
Weekly alternative data,18,10.8
Daily alternative data,14,6.4


,matches BLS series,MoM corr,basket weight %
leaf,,,
170,Medical services,1.000,7.478
96,OER,1.000,5.515
111,Telephone,0.978,2.762
135,New vehicles,1.000,1.898
156,Vehicle maintenance,1.000,1.597
182,Food away,1.000,1.490
128,Water/sewer,1.000,1.333
146,Electricity,1.000,1.316
87,Tuition,1.000,1.310


In [5]:
# Published Truflation vs its two halves: the BLS-mirror part and the alternative-data part (current vintage)
d = pd.concat({"Truflation (published)": inp.truf_pub, "Alternative-data part": vint_c["Ex-BLS headline"],
               "BLS-mirror part": vint_c["BLS-mirror part"]}, axis=1).loc["2019-01-01":]
cpi_y = res.cpi["cpi_yoy"].loc["2019-01-01":]
fig = go.Figure()
for name, col, w in [("Truflation (published)", ch.C_TRUF, 2), ("Alternative-data part", "#e6a23c", 1.4), ("BLS-mirror part", "#9085e9", 1.4)]:
    fig.add_trace(go.Scatter(x=d.index, y=d[name], name=name, line=dict(color=col, width=w), hovertemplate="%{y:.2f}%<extra>" + name + "</extra>"))
fig.add_trace(go.Scatter(x=cpi_y.index, y=cpi_y, name="Official CPI YoY", line=dict(color=ch.C_CPI, width=2.2, shape="hv"),
                         hovertemplate="%{y:.2f}%<extra>CPI</extra>"))
ch.add_recessions(fig, res.recessions); ch.add_shutdown_gap(fig)
lay = ch.base_layout("Truflation's two halves vs CPI (YoY %)", 480)
lay["legend"].update(y=-0.12, yanchor="top", x=0.5, xanchor="center"); lay["margin"].update(b=90)
lay["yaxis"].update(ticksuffix="%"); fig.update_layout(**lay); fig.show()
last = d.dropna().iloc[-1]
display(Markdown(f"Latest: alternative-data part **{last['Alternative-data part']:.2f}%**, BLS-copy part "
                 f"**{last['BLS-mirror part']:.2f}%**, CPI **{cpi_y.dropna().iloc[-1]:.2f}%**."))

Latest: alternative-data part **2.95%**, BLS-copy part **3.36%**, CPI **3.74%**.

## Truflation vs CPI

In [6]:
mm = res.monthly[["truf_avg", "cpi_yoy", "spread"]].dropna()
dd = mm[["truf_avg", "cpi_yoy"]].diff()
lags = pd.Series({k: dd["truf_avg"].shift(k).corr(dd["cpi_yoy"]) for k in range(-3, 4)}, name="corr of monthly changes")
lags.index.name = "Truflation lag (months; + = Truflation leads)"
stats = ch.regime_stats(res.monthly)

fig = make_subplots(rows=1, cols=2, column_widths=[0.4, 0.6], horizontal_spacing=0.1,
                    subplot_titles=("Lead/lag correlation", "Spread (pp) & divergence episodes"))
fig.add_trace(go.Bar(x=lags.index, y=lags.values, marker_color=[ch.C_TRUF if k == 0 else ch.C_NEUTRAL for k in lags.index],
                     hovertemplate="lag %{x}: %{y:.2f}<extra></extra>", showlegend=False), 1, 1)
fig.add_trace(go.Bar(x=mm.index, y=mm["spread"], marker_color=ch.C_NEUTRAL, name="Monthly spread",
                     hovertemplate="%{x|%b-%y}: %{y:+.3f}pp<extra></extra>", showlegend=False), 1, 2)
# divergence episodes: >= 3 consecutive months with |spread| > 0.75pp
big = (mm["spread"].abs() > 0.75)
runs = (big != big.shift()).cumsum()[big]
episodes = []
for _, g in mm[big].groupby(runs):
    if len(g) >= 3:
        episodes.append({"start": g.index[0], "end": g.index[-1], "months": len(g), "mean spread (pp)": g["spread"].mean(),
                         "peak |spread| (pp)": g["spread"].abs().max()})
        fig.add_vrect(x0=g.index[0], x1=g.index[-1] + pd.offsets.MonthEnd(0), fillcolor="rgba(255,75,75,0.10)", line_width=0, row=1, col=2)
lay = ch.base_layout("", 380); lay.pop("xaxis"); lay.pop("yaxis"); lay["hovermode"] = "closest"; lay["margin"].update(t=40)
fig.update_layout(**lay); fig.update_yaxes(gridcolor=ch.GRID_COLOR); fig.update_xaxes(gridcolor=ch.GRID_COLOR); fig.show()
ep = pd.DataFrame(episodes)
ep["start"], ep["end"] = ep["start"].dt.strftime("%b-%y"), ep["end"].dt.strftime("%b-%y")
display(ep.style.format({"mean spread (pp)": "{:+.3f}", "peak |spread| (pp)": "{:.3f}"}).hide(axis="index"))
display(stats.style.format(ch.STATS_FORMAT))

start,end,months,mean spread (pp),peak |spread| (pp)
Apr-11,Jun-11,3,+1.174,1.247
Mar-21,Sep-22,19,+2.422,3.544
Apr-23,Feb-24,11,-1.037,1.313
Jul-25,Sep-25,3,-0.895,0.969
Jan-26,Sep-26,9,-1.556,2.210


,months,corr (levels),corr (1m changes),mean spread (pp),MAE (pp),max |spread| (pp)
Full sample,188,0.94,0.84,+0.087,0.653,3.544
2010–2019 (low inflation),108,0.93,0.78,+0.078,0.316,1.247
2020–2022 (COVID & surge),36,0.98,0.92,+1.264,1.417,3.544
2023–2024 (disinflation),24,0.95,0.91,-0.696,0.701,1.313
2025– (tariffs),20,0.28,0.56,-1.042,1.042,2.210


No lead: changes move together (≈0.84) but the level gap shifts by regime.

## Method
Forecast = last print + base effect + change in Truflation components. Walk-forward since 2014, day 19 of each month, point-in-time data only.

## Backtest (walk-forward, 2014–)

In [7]:
cols = ["n", "RMSE (pp)", "MAE (pp)", "Bias (pp)", "RMSE vs naive", "Direction hit", "Within ±0.1pp", "DM p vs naive", "DM p vs base effect"]
fmt = {"n": "{:.0f}", "RMSE (pp)": "{:.3f}", "MAE (pp)": "{:.3f}", "Bias (pp)": "{:+.3f}", "RMSE vs naive": "{:.2f}",
       "Direction hit": "{:.0%}", "Within ±0.1pp": "{:.0%}", "DM p vs naive": "{:.3f}", "DM p vs base effect": "{:.3f}"}
for name, per in nc.PERIODS.items():
    display(Markdown(f"**Origin day 19 · {name}**"))
    display(nc.scorecard(bt19, per)[cols].style.format(fmt, na_rep="–")
            .background_gradient(subset=["RMSE vs naive"], cmap="RdYlGn_r", vmin=0.3, vmax=1.1))
display(Markdown("**Robustness: origin on the eve of the release (full month of Truflation), Truflation live period**"))
display(nc.scorecard(btev, nc.PERIODS["Truflation live (Dec-2021–)"])[cols].style.format(fmt, na_rep="–"))

**Origin day 19 · Full OOS (2014–)**

,n,RMSE (pp),MAE (pp),Bias (pp),RMSE vs naive,Direction hit,Within ±0.1pp,DM p vs naive,DM p vs base effect
Naive (last print),150,0.394,0.292,-0.012,1.00,–,25%,–,–
Base effect,150,0.339,0.249,-0.024,0.86,69%,30%,0.111,–
AR momentum,149,0.307,0.231,-0.026,0.78,77%,26%,0.009,0.046
Truflation bridge,150,0.240,0.178,-0.017,0.61,84%,39%,0.000,0.000
Truflation components,150,0.180,0.132,+0.002,0.46,90%,52%,0.000,0.000
Bridge on published series*,150,0.219,0.164,-0.017,0.56,85%,40%,0.000,0.000
"Components, current vintage*",150,0.163,0.119,-0.012,0.41,91%,54%,0.000,0.000


**Origin day 19 · Pre-launch backfill (2014–2021)**

,n,RMSE (pp),MAE (pp),Bias (pp),RMSE vs naive,Direction hit,Within ±0.1pp,DM p vs naive,DM p vs base effect
Naive (last print),93,0.387,0.279,-0.053,1.00,–,24%,–,–
Base effect,93,0.326,0.242,-0.035,0.84,68%,31%,0.130,–
AR momentum,93,0.283,0.218,-0.053,0.73,76%,25%,0.025,0.034
Truflation bridge,93,0.233,0.173,-0.023,0.60,84%,39%,0.006,0.001
Truflation components,93,0.146,0.102,-0.015,0.38,91%,63%,0.001,0.000
Bridge on published series*,93,0.191,0.144,-0.008,0.49,86%,47%,0.004,0.001
"Components, current vintage*",93,0.150,0.105,-0.019,0.39,91%,60%,0.002,0.000


**Origin day 19 · Truflation live (Dec-2021–)**

,n,RMSE (pp),MAE (pp),Bias (pp),RMSE vs naive,Direction hit,Within ±0.1pp,DM p vs naive,DM p vs base effect
Naive (last print),57,0.405,0.312,+0.054,1.00,–,26%,–,–
Base effect,57,0.360,0.260,-0.006,0.89,72%,28%,0.467,–
AR momentum,56,0.342,0.253,+0.017,0.85,77%,28%,0.180,0.472
Truflation bridge,57,0.252,0.186,-0.009,0.62,84%,39%,0.007,0.010
Truflation components,57,0.224,0.181,+0.029,0.55,88%,33%,0.003,0.034
Bridge on published series*,57,0.258,0.198,-0.032,0.64,82%,28%,0.005,0.046
"Components, current vintage*",57,0.182,0.142,+0.001,0.45,91%,44%,0.000,0.011


**Origin day 19 · Since 2025**

,n,RMSE (pp),MAE (pp),Bias (pp),RMSE vs naive,Direction hit,Within ±0.1pp,DM p vs naive,DM p vs base effect
Naive (last print),20,0.348,0.262,-0.042,1.00,–,35%,–,–
Base effect,20,0.326,0.244,+0.085,0.94,75%,25%,0.721,–
AR momentum,19,0.337,0.227,+0.075,0.97,79%,30%,0.779,0.900
Truflation bridge,20,0.216,0.158,+0.020,0.62,85%,40%,0.065,0.017
Truflation components,20,0.160,0.128,+0.046,0.46,95%,55%,0.039,0.038
Bridge on published series*,20,0.310,0.236,-0.042,0.89,65%,30%,0.440,0.738
"Components, current vintage*",20,0.148,0.123,+0.021,0.42,90%,45%,0.029,0.033


**Robustness: origin on the eve of the release (full month of Truflation), Truflation live period**

,n,RMSE (pp),MAE (pp),Bias (pp),RMSE vs naive,Direction hit,Within ±0.1pp,DM p vs naive,DM p vs base effect
Naive (last print),57,0.405,0.312,+0.054,1.00,–,26%,–,–
Base effect,57,0.360,0.260,-0.006,0.89,72%,28%,0.467,–
AR momentum,56,0.342,0.253,+0.026,0.85,77%,30%,0.185,0.446
Truflation bridge,57,0.255,0.191,-0.017,0.63,89%,39%,0.007,0.013
Truflation components,57,0.212,0.169,+0.019,0.52,91%,30%,0.002,0.027
Bridge on published series*,57,0.248,0.193,-0.018,0.61,88%,30%,0.003,0.033
"Components, current vintage*",57,0.187,0.145,+0.003,0.46,89%,46%,0.000,0.012


In [8]:
# Forecast vs actual, and rolling 12-month RMSE, for the main models
show = ["Naive (last print)", "Base effect", "Truflation bridge", BEST]
colors = {"Naive (last print)": ch.C_NEUTRAL, "Base effect": "#9085e9", "Truflation bridge": "#e6a23c", BEST: ch.C_TRUF}
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.6, 0.4], vertical_spacing=0.08,
                    subplot_titles=("Out-of-sample nowcasts (made on day 19) vs the actual print", "Rolling 12-month RMSE (pp)"))
fig.add_trace(go.Scatter(x=bt19.index, y=bt19["y"], name="Actual CPI YoY", line=dict(color=ch.C_CPI, width=2.4),
                         hovertemplate="%{y:.2f}%<extra>Actual</extra>"), 1, 1)
for mname in ["Naive (last print)", BEST]:
    fig.add_trace(go.Scatter(x=bt19.index, y=bt19[mname], name=mname, mode="lines", line=dict(color=colors[mname], width=1.4, dash="dot" if "Naive" in mname else None),
                             hovertemplate="%{y:.2f}%<extra>" + mname + "</extra>"), 1, 1)
for mname in show:
    r = ((bt19[mname] - bt19["y"]) ** 2).rolling(12, min_periods=9).mean() ** 0.5
    fig.add_trace(go.Scatter(x=r.index, y=r, name=f"RMSE · {mname}", line=dict(color=colors[mname], width=1.6),
                             showlegend=True, hovertemplate="%{y:.3f}pp<extra>" + mname + "</extra>"), 2, 1)
fig.add_vline(x=pd.Timestamp("2021-12-01"), line=dict(color=ch.TEXT_MUTED, dash="dash", width=1))
fig.add_annotation(x=pd.Timestamp("2021-12-01"), y=1, yref="paper", text="Truflation live →", showarrow=False, xanchor="left",
                   font=dict(size=10, color=ch.TEXT_MUTED))
lay = ch.base_layout("", 640); lay.pop("xaxis"); lay.pop("yaxis")
lay["legend"].update(y=-0.08, yanchor="top", x=0.5, xanchor="center")
fig.update_layout(**lay); fig.update_xaxes(gridcolor=ch.GRID_COLOR); fig.update_yaxes(gridcolor=ch.GRID_COLOR)
fig.update_yaxes(ticksuffix="%", row=1, col=1); fig.show()

In [9]:
# What the components model has learned (fitted on everything released before today's origin)
train = ft19[ft19.index.map(inp.release) < row["origin"]].dropna(subset=["dy"])
mdl = [m for m in nc.model_suite() if m.name == BEST][0]
mdl.fit(train[mdl.features], train["dy"])
coef = mdl.coefs().rename({"intercept": "Intercept", "BE": "Base effect (pp per pp)", "dFood": "Δ Truflation food (pp per pp)",
                           "dEnergy": "Δ Truflation energy", "dHousing": "Δ Truflation housing", "dRest": "Δ Truflation rest"})
display(coef.to_frame("coefficient").style.format("{:+.3f}"))

,coefficient
Intercept,+0.011
Base effect (pp per pp),+0.189
Δ Truflation food (pp per pp),+0.054
Δ Truflation energy,+0.140
Δ Truflation housing,+0.110
Δ Truflation rest,+0.120


Truflation models beat all baselines. \* = Truflation's published history (contains later data).

## October 2026 nowcast

In [10]:
tbl = pd.DataFrame({k: v["pred"] for k, v in live.items()}).loc[nc.MAIN_MODELS + nc.DIAGNOSTICS]
tbl.columns = ["Oct-25 interpolated (central)", "Sep-25 carried forward", "Nov-25 carried back"]
display(Markdown(f"Origin {row['origin']:%d %b %Y} (day {row['days']} of October) · last print {row['L']:%b-%y} = {row['yL']:.2f}%"))
display(tbl.style.format("{:.2f}%"))

# waterfall of the central components nowcast
steps = contrib.drop("Model intercept")
fig = go.Figure(go.Waterfall(x=["Sep-26 print", "Intercept"] + list(steps.index) + ["Oct-26 nowcast"],
                             measure=["absolute", "relative"] + ["relative"] * len(steps) + ["total"],
                             y=[row["yL"], contrib["Model intercept"]] + list(steps.values) + [0],
                             text=[f"{row['yL']:.2f}%", f"{contrib['Model intercept']:+.2f}"] + [f"{v:+.2f}" for v in steps.values] + [f"{p:.2f}%"],
                             textposition="outside", connector=dict(line=dict(color=ch.GRID_COLOR)),
                             increasing=dict(marker=dict(color=ch.C_TRUF)), decreasing=dict(marker=dict(color=ch.C_CPI)),
                             totals=dict(marker=dict(color="#e6e9ef"))))
lay = ch.base_layout("Oct-26 nowcast build-up (components model)", 420)
lay["yaxis"].update(range=[row["yL"] - 0.15, p + 0.2], ticksuffix="%"); lay["hovermode"] = "closest"
fig.update_layout(**lay, showlegend=False); fig.show()

Origin 19 Oct 2026 (day 19 of October) · last print Sep-26 = 3.74%

,Oct-25 interpolated (central),Sep-25 carried forward,Nov-25 carried back
Naive (last print),3.74%,3.74%,3.74%
Base effect,4.08%,4.00%,4.17%
AR momentum,4.11%,4.05%,4.16%
Truflation bridge,4.00%,3.97%,4.04%
Truflation components,3.99%,3.98%,4.01%
Bridge on published series*,4.16%,4.14%,4.18%
"Components, current vintage*",4.26%,4.25%,4.28%


In [11]:
# context: the last two years of CPI with the nowcast and its empirical 80% band
hist = res.cpi["cpi_yoy"].loc["2024-10-01":]
fig = go.Figure()
fig.add_trace(go.Scatter(x=hist.index, y=hist, name="CPI YoY (printed)", mode="lines+markers", line=dict(color=ch.C_CPI, width=2.2),
                         hovertemplate="%{x|%b-%y}: %{y:.2f}%<extra></extra>"))
t = pd.Timestamp("2026-10-01")
fig.add_trace(go.Scatter(x=[t, t], y=[p + q_lo, p + q_hi], mode="lines", line=dict(color=ch.C_TRUF, width=10), opacity=0.35,
                         name="80% band (live-period errors)", hoverinfo="skip"))
fig.add_trace(go.Scatter(x=[t], y=[p], mode="markers+text", marker=dict(symbol="diamond", size=14, color=ch.C_TRUF),
                         text=[f"{p:.2f}%  "], textposition="middle left", name="Oct-26 nowcast",
                         hovertemplate=f"Oct-26 nowcast {p:.2f}% (80%: {p+q_lo:.2f}–{p+q_hi:.2f}%)<extra></extra>"))
for k, v in live.items():
    if k != CENTRAL:
        fig.add_trace(go.Scatter(x=[t], y=[v["pred"][BEST]], mode="markers", marker=dict(symbol="line-ew-open", size=16, color="#e6e9ef"),
                                 name="Other Oct-25 base scenarios", showlegend=k == "Sep-25 carried forward",
                                 hovertemplate=f"{k}: %{{y:.2f}}%<extra></extra>"))
ch.add_shutdown_gap(fig)
lay = ch.base_layout("Next print in context", 420); lay["yaxis"].update(ticksuffix="%"); lay["hovermode"] = "closest"
lay["legend"].update(y=-0.12, yanchor="top", x=0.5, xanchor="center"); lay["margin"].update(b=90)
lay["xaxis"].update(range=[pd.Timestamp("2024-09-15"), pd.Timestamp("2026-12-15")])
fig.update_layout(**lay); fig.show()

The missing Oct-2025 base barely matters (3.98–4.01%).

## Caveats
Point-in-time emulated by rule · small live sample · ~30% of Truflation stalls when BLS does.